In [16]:
# Step 1: Import Libraries and Load the Model
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import load_model

In [17]:

# Load the IMDB dataset word index
word_index = imdb.get_word_index()
reverse_word_index = {value: key for key, value in word_index.items()}
max_len = 250

In [18]:
# Load the pre-trained LSTM model
model = load_model('simple_rnn_imdb.h5', compile=False)
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 250, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,411,713 (5.39 MB)

 Trainable params: 1,411,713 (5.39 MB)

 Non-trainable params: 0 (0.00 B)

In [19]:
model.get_weights()

[array([[-9.77538303e-02,  2.43218362e-01, -1.11890724e-02, ...,
         -6.19153343e-02,  1.01275884e-01,  3.64948210e-05],
        [ 9.04740021e-02,  1.18394062e-01,  1.15694031e-01, ...,
          8.00441802e-02, -1.66187715e-02, -1.08402655e-01],
        [-7.37688364e-03, -4.27630655e-02, -1.08878827e-02, ...,
         -7.49123096e-02, -7.04251081e-02,  1.01635363e-02],
        ...,
        [ 1.15815494e-02,  1.14602922e-02, -8.33269209e-02, ...,
         -2.44347408e-04,  4.78058122e-02,  4.37542424e-03],
        [-3.80611531e-02,  6.43405095e-02,  4.08221520e-02, ...,
         -3.67735997e-02,  4.21186984e-02, -1.09195495e-02],
        [-3.85183282e-02, -7.59566277e-02, -2.13254355e-02, ...,
         -8.88663828e-02,  9.79805067e-02,  6.27737343e-02]],
       shape=(10000, 128), dtype=float32),
 array([[ 0.114553  ,  0.03662785, -0.12347922, ...,  0.05667416,
         -0.00294457,  0.09891006],
        [-0.15667473,  0.03861661, -0.02384593, ..., -0.14708841,
          0.1119506

In [20]:
# Step 2: Helper Functions
# Function to decode reviews
def decode_review(encoded_review):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review])

# Function to preprocess user input
def preprocess_text(text):
    words = [w.strip('.,!?;:\"()[]') for w in text.lower().split()]
    encoded_review = [word_index.get(word, 2) + 3 for word in words if word]
    if not encoded_review:
        encoded_review = [2]

    padded_review = sequence.pad_sequences([encoded_review], maxlen=max_len)
    return padded_review

In [21]:
### Prediction  function

def predict_sentiment(review):
    preprocessed_input = preprocess_text(review)
    prediction = model.predict(preprocessed_input, verbose=0)
    raw_score = prediction[0][0]
    score = float(np.nan_to_num(raw_score, nan=0.0, posinf=1.0, neginf=0.0))
    sentiment = 'Positive' if score > 0.5 else 'Negative'
    return sentiment, score



In [22]:
# Step 4: User Input and Prediction
# Example review for prediction
example_review = "This movie was fantastic! The acting was great and the plot was thrilling."

sentiment,score=predict_sentiment(example_review)

print(f'Review: {example_review}')
print(f'Sentiment: {sentiment}')
print(f'Prediction Score: {score}')

Review: This movie was fantastic! The acting was great and the plot was thrilling.
Sentiment: Positive
Prediction Score: 0.8949947953224182
